In [8]:
import json
import ipywidgets as widgets
from IPython.display import display
import ipyvuetify as v
from ipyvuetify import VuetifyTemplate

# Load eligible players
with open('eligible_players.json') as f:
    players = json.load(f)['players']

# Initialize selected players structure
selected_players = {
    'C': {'id': '', 'name': '', 'lineup_slot': ''},
    '1B': {'id': '', 'name': '', 'lineup_slot': ''},
    '2B': {'id': '', 'name': '', 'lineup_slot': ''},
    '3B': {'id': '', 'name': '', 'lineup_slot': ''},
    'SS': {'id': '', 'name': '', 'lineup_slot': ''},
    'OF': [{'id': '', 'name': '', 'lineup_slot': ''} for _ in range(3)],
    'DH': {'id': '', 'name': '', 'lineup_slot': ''},
    'PH': [{'id': '', 'name': '', 'lineup_slot': '', 'bench_order': ''} for _ in range(6)]
}

# Build dropdown options by position, with DH including every player\positions = ['C', '1B', '2B', '3B', 'SS'] + ['OF'] * 3 + ['DH']
options = {pos: ['Select Player'] for pos in positions}
for p in players:
    pos = p['pos']
    if pos in options:
        options[pos].append(p['longName'])
    else:
        options['DH'].append(p['longName'])
bench_opts = ['Select Player'] + [p['longName'] for p in players if p['pos'] != 'P']

# Create dropdown widgets
dropdowns = [widgets.Dropdown(options=options[pos], description=pos) for pos in positions]
bench_dropdowns = [widgets.Dropdown(options=bench_opts, description=f'PH{i+1}') for i in range(6)]
select_btn = widgets.Button(description='Select team')
select_out = widgets.Output()

def select_team(b):
    select_out.clear_output()
    chosen = [d.value for d in dropdowns] + [d.value for d in bench_dropdowns]
    if 'Select Player' in chosen:
        with select_out:
            print('Error: All slots must be filled.')
        return
    if len(chosen) != len(set(chosen)):
        with select_out:
            print('Error: Duplicate player selected.')
        return
    of_idx = 0
    for d in dropdowns:
        name = d.value
        pid = next(p['playerID'] for p in players if p['longName'] == name)
        pos = d.description
        if pos == 'OF':
            selected_players['OF'][of_idx].update({'id': pid, 'name': name})
            of_idx += 1
        else:
            selected_players[pos].update({'id': pid, 'name': name})
    for i, d in enumerate(bench_dropdowns):
        name = d.value
        pid = next(p['playerID'] for p in players if p['longName'] == name)
        selected_players['PH'][i].update({'id': pid, 'name': name, 'position': 'PH', 'bench_order': i+1})
    with select_out:
        print(selected_players)

select_btn.on_click(select_team)
# Display selection UI
ui_items = []
for d in dropdowns + bench_dropdowns:
    ui_items.append(d)
    ui_items.append(v.Html(tag='br'))
ui_items.append(select_btn)
ui_items.append(select_out)
display(*ui_items)

# Prepare drag-and-drop lineup widget using vuedraggable

def lineup_items():
    items = []
    for k, v in selected_players.items():
        if k == 'OF':
            items += [p['name'] for p in v if p['name']]
        elif k != 'PH' and v['name']:
            items.append(v['name'])
    return items

template = '''
<draggable v-model="items" class="list-group" ghost-class="ghost" :options="{group:'lineup', animation:200}">
  <v-list two-line>
    <v-list-item v-for="(item, idx) in items" :key="item">
      <v-list-item-content>{{ idx + 1 }}: {{ item }}</v-list-item-content>
    </v-list-item>
  </v-list>
</draggable>
'''

# Initialize Vue template with data as JSON string
lineup_drag = VuetifyTemplate(
    template=template,
    components={'draggable': 'vuedraggable'},
    data=json.dumps({'items': lineup_items()})
)

update_btn = widgets.Button(description='Update lineup')
update_out = widgets.Output()

def update_lineup(b):
    update_out.clear_output()
    # Parse updated order from data string
    model = json.loads(lineup_drag.data)
    new_order = model.get('items', [])
    # Reset slots
    for k, v in selected_players.items():
        if isinstance(v, list):
            for p in v:
                p['lineup_slot'] = ''
        else:
            v['lineup_slot'] = ''
    # Assign new slots
    for idx, name in enumerate(new_order):
        for k, v in selected_players.items():
            if k == 'OF':
                for p in v:
                    if p['name'] == name:
                        p['lineup_slot'] = idx + 1
            elif k != 'PH' and v['name'] == name:
                v['lineup_slot'] = idx + 1
    with update_out:
        print('Updated lineup:')
        print(selected_players)

update_btn.on_click(update_lineup)
# Display drag-and-drop UI
display(v.Html(tag='h4', children=['Drag to order batting lineup']))
display(lineup_drag, update_btn, update_out)


Dropdown(description='C', options=('Select Player', 'Jacob Stallings', 'Henry Davis', 'Jonah Heim', 'Austin Ba…

Html(tag='br')

Dropdown(description='1B', options=('Select Player', 'Kody Clemens', 'Justin Turner', 'Paul Goldschmidt', 'Jon…

Html(tag='br')

Dropdown(description='2B', options=('Select Player', 'Nicky Lopez', 'Gavin Lux', 'Amed Rosario', 'Jonathan Ind…

...

Output()

Html(children=['Drag to order batting lineup'], tag='h4')

VuetifyTemplate(components={'draggable': 'vuedraggable'}, data='{"items": []}', template='\n<draggable v-model…

Button(description='Update lineup', style=ButtonStyle())

Output()